# Monolith assistant: LoRA fine-tune (two models)
Runtime → Change runtime type → **T4 GPU**. Pick `RUN` in the second cell, run top to bottom, then repeat for the other.

| `RUN` | Model | Role | Training |
| --- | --- | --- | --- |
| `text` | `LiquidAI/LFM3-1.7B` | the main agent: answers and action lines, 4-bit (about 1 GB) | QLoRA (4-bit base, fp16 adapters) |
| `vl` | `LiquidAI/LFM2.5-VL-450M` | reads pictures; small enough to stay loaded | LoRA, fp32 |

Both train on the same data and are scored by the same checker, so the numbers compare directly. Written against the model cards
(LFM3 needs `trust_remote_code`); not yet run on a GPU, so expect to fix a cell or two.

In [ ]:
!pip -q install -U transformers peft accelerate bitsandbytes pillow
!git clone -q https://github.com/nicosbrooks/Monolith.git -b ui-glass-rework
%cd Monolith/training
!python -m pytest -q tests

In [ ]:
RUN = 'text'   # 'text' = LFM3-1.7B (4-bit)   |   'vl' = LFM2.5-VL-450M
MODEL = {'text': 'LiquidAI/LFM3-1.7B', 'vl': 'LiquidAI/LFM2.5-VL-450M'}[RUN]
FOUR = '--four-bit' if RUN == 'text' else ''
OUT = f'out_{RUN}'
print(MODEL, OUT)

## 1. Data
Synthetic, in the app's exact prompt format; the test split uses phrasings and titles training never sees. Picture examples are only used by the `vl` run.

In [ ]:
!python generate_data.py --n 6000 --out data --images
import json; rows=[json.loads(l) for l in open('data/train.jsonl')]; print(len(rows)); print(rows[0]['messages'][1]['content']); print(rows[0]['messages'][2]['content'])

## 2. Baseline
The untuned model with the app's ACT_GUIDE. This is the score to beat. (LFM3 thinks by default; the evaluator turns that off, as the app will.)

In [ ]:
!python evaluate.py --model $MODEL $FOUR --data data/test.jsonl --guide --limit 200

## 3. Train (assistant tokens only)

In [ ]:
import json, torch
from PIL import Image
from transformers import AutoModelForCausalLM, AutoModelForImageTextToText, AutoProcessor, AutoTokenizer, BitsAndBytesConfig, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

VL = RUN == 'vl'
if VL:
    proc = AutoProcessor.from_pretrained(MODEL); tok = proc.tokenizer
    model = AutoModelForImageTextToText.from_pretrained(MODEL, torch_dtype=torch.float32, device_map={'':0})
    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules='all-linear', exclude_modules=r'.*(vision|siglip|projector).*', task_type='CAUSAL_LM')
else:
    tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=q, device_map={'':0}, trust_remote_code=True)
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules='all-linear', task_type='CAUSAL_LM')
model = get_peft_model(model, lora); model.print_trainable_parameters()
tok.padding_side = 'right'
if tok.pad_token is None: tok.pad_token = tok.eos_token

def vl_msgs(row, upto):
    msgs = [dict(m) for m in row['messages'][:upto]]
    if row.get('image'):
        u = [m for m in msgs if m['role']=='user'][-1]; u['content'] = [{'type':'image'},{'type':'text','text':u['content']}]
    return [m if isinstance(m['content'], list) else {'role':m['role'],'content':[{'type':'text','text':m['content']}]} for m in msgs]

def template(msgs, gen):
    if VL: return proc.apply_chat_template(msgs, add_generation_prompt=gen, tokenize=False)
    return tok.apply_chat_template(msgs, add_generation_prompt=gen, tokenize=False, enable_thinking=False)

class Data(torch.utils.data.Dataset):
    def __init__(s, path):
        s.rows=[json.loads(l) for l in open(path)]
        if not VL: s.rows=[r for r in s.rows if not r.get('image')]   # a text model can't see pictures
    def __len__(s): return len(s.rows)
    def __getitem__(s, i): return s.rows[i]

def collate(batch):
    imgs = [Image.open(r['image']).convert('RGB') for r in batch if r.get('image')] if VL else []
    get = (lambda r,n: vl_msgs(r,n)) if VL else (lambda r,n: r['messages'][:n])
    prefixes = [template(get(r,2), True) for r in batch]
    texts = [template(get(r,3), False) for r in batch]
    if VL:
        enc = proc(text=texts, images=imgs or None, return_tensors='pt', padding=True)
        pre = proc(text=prefixes, images=imgs or None, return_tensors='pt', padding=True)
    else:
        enc = tok(texts, return_tensors='pt', padding=True, add_special_tokens=False)
        pre = tok(prefixes, return_tensors='pt', padding=True, add_special_tokens=False)
    labels = enc['input_ids'].clone()
    for i in range(len(batch)): labels[i, :int(pre['attention_mask'][i].sum())] = -100   # reply only (right padding)
    labels[enc['attention_mask']==0] = -100
    enc['labels'] = labels
    return enc

args = TrainingArguments(output_dir=OUT, per_device_train_batch_size=4 if VL else 2, gradient_accumulation_steps=8 if VL else 16,
    num_train_epochs=2, learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.05, logging_steps=10, save_strategy='epoch',
    fp16=not VL, bf16=False, remove_unused_columns=False, report_to='none', gradient_checkpointing=True,
    optim='paged_adamw_8bit' if not VL else 'adamw_torch')
Trainer(model=model, args=args, train_dataset=Data('data/train.jsonl'), data_collator=collate).train()
model.save_pretrained(f'{OUT}/lora'); tok.save_pretrained(f'{OUT}/lora')

If memory runs out, halve the batch size and double the accumulation. Check the printed loss falls; if it stays flat the label mask is wrong (print a decoded `labels != -100` slice for one batch).

In [ ]:
del model; torch.cuda.empty_cache()
!python evaluate.py --model $MODEL $FOUR --adapter $OUT/lora --data data/test.jsonl --limit 300

## 4. Export
Merge the adapter into **full-precision** weights, re-run the evaluator on the merged folder, then quantize.
| Target | Text model (LFM3-1.7B) | Picture model (LFM2.5-VL-450M) |
| --- | --- | --- |
| Web | ONNX (Transformers.js) or GGUF via llama.cpp WASM; the model card lists no LiteRT-LM build | LiteRT-LM (`litert-community/LFM2.5-VL-450M`, int4 about 0.41 GB) |
| iOS | MLX 4-bit (`LFM3-1.7B-MLX-4bit` shows the layout) | MLX |
| Desktop | GGUF q4 | GGUF |
Quantize to 4-bit **after** merging, then score the 4-bit file: quantization can lose points the full-precision score hid.

In [ ]:
from peft import PeftModel
Auto = AutoModelForImageTextToText if VL else AutoModelForCausalLM
base = Auto.from_pretrained(MODEL, torch_dtype=torch.float16, trust_remote_code=True)
merged = PeftModel.from_pretrained(base, f'{OUT}/lora').merge_and_unload()
merged.save_pretrained(f'{OUT}/merged'); tok.save_pretrained(f'{OUT}/merged')
!ls -lh $OUT/merged